In [1]:
### RAG pipeline - Data Ingestion to Vector DB pipline
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/xg/4zjjxl1j4lg2q4_2ly46c34w0000gn/T/ipykernel_8562/401592239.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
/Users/qmc/Neutrons/rag/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

def process_all_txts(txt_directory):
    """Process all .txt files in a directory"""
    all_documents = []
    txt_dir = Path(txt_directory)

    # Find all .txt files recursively
    txt_files = list(txt_dir.glob("**/*.txt"))

    print(f"Found {len(txt_files)} txt files to process")

    for txt_file in txt_files:
        print(f"\nProcessing: {txt_file.name}")
        try:
            loader = TextLoader(str(txt_file), encoding="utf-8", autodetect_encoding=True)
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = txt_file.name
                doc.metadata['file_type'] = 'txt'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} document(s)")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all .txt files in the data directory
all_txt_documents = process_all_txts("../data")

Found 1 txt files to process

Processing: triple_axis_scan.txt
  ✓ Loaded 1 document(s)

Total documents loaded: 1


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_documents = process_all_pdfs("../data") + process_all_txts("../data")

Found 7 PDF files to process

Processing: Staica 1975 Part1.pdf
  ✓ Loaded 4 pages

Processing: Popvici 1975 part4.pdf
  ✓ Loaded 7 pages

Processing: Cooper-Nathans original 1967.pdf
  ✓ Loaded 11 pages

Processing: Popvici and stoica 1975 part3.pdf
  ✓ Loaded 4 pages

Processing: Stoica 1975 part2.pdf
  ✓ Loaded 4 pages

Processing: Werner and Pynn 1971.pdf
  ✓ Loaded 15 pages

Processing: Mitchell - 1984.pdf
  ✓ Loaded 9 pages

Total documents loaded: 54
Found 1 txt files to process

Processing: triple_axis_scan.txt
  ✓ Loaded 1 document(s)

Total documents loaded: 1


In [4]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
chunks=split_documents(all_documents)

Split 55 documents into 289 chunks

Example chunk:
Content: 189 
Acta Cryst. (1975). A31, 189 
On the Resolution of Slow-Neutron Spectrometers. 
I. A General Method to Calculate Resolution Functions 
By A. D. STOICA 
Institute for Atomic Physics, Bucharest, P....
Metadata: {'producer': 'PageGenie PDFGenerator; modified using iText 4.2.0 by 1T3XT', 'creator': 'a11676.pdf', 'creationdate': 'Wed May 16 23:41:23 2001', 'keywords': '', 'moddate': '2026-04-06T09:46:25-07:00', 'subject': 'Acta Crystallographica Section A 1975.31:189-192', 'author': '', 'title': 'On the resolution of slow‐neutron spectrometers. I. A general method to calculate resolution functions', 'source': '../data/pdf/Staica 1975 Part1.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Staica 1975 Part1.pdf', 'file_type': 'pdf'}


In [5]:
### embedding and vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity



In [7]:
class EmbedingManager:
    """Hanldes document embedding generation using SentenceTransformer."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize embedding manager."""
        self.model_name = model_name
        self.model = None
        self._load_model() 
    
    def _load_model(self):
        try:
            print(f"Loading embeding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]):
        if not self.model:
            raise ValueError("Model not loaded.")
        print(f"generating embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings

embedding_manager = EmbedingManager()
embedding_manager

Loading embeding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15833.94it/s]


Model loaded successfully. Embedding dimension: 384


In [8]:
### Vector Store

class VectorStore:
    """Manage document embeddings in a chromaDB vector store."""
    def __init__(self, collection_name: str = "pdf_documents", persistent_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persistent_directory, exist_ok =True)
            self.client = chromadb.PersistentClient(path = self.persistent_directory)
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialied. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            raise ValueError(e)
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store....")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
        except Exception as e:
            raise ValueError("e")
vectorstore = VectorStore()

Vector store initialied. Collection: pdf_documents
Existing documents in collection: 2218


In [9]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

generating embedding for 289 texts...


Batches: 100%|██████████| 10/10 [00:00<00:00, 12.35it/s]


Generated embeddings with shape : (289, 384)
Adding 289 documents to vector store....


In [10]:
### Retriever pipeline from VectorStore

class RAGRetriever:
    """Handles query-based retrieval from the vector store."""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbedingManager):
        """Init."""
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> list[Dict[str, Any]]:
        """Retrieve relevant documents for a query."""
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found.")
            return retrieved_docs
        except Exception as e:
            print(e)  
            return []

rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager= embedding_manager)

In [11]:
### simple rag pipline
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(model="gemma4:latest", temperature=0.1, max_completion_tokens=1024)

In [12]:
### Q&A chatbot — answer general triple-axis questions from the knowledge base

QA_SYSTEM_PROMPT = """You are a neutron scattering assistant that answers questions about \
Triple Axis Spectrometers (TAS): instrument geometry, resolution functions \
(Cooper-Nathans, Popovici), reciprocal-space scans, and data collection.

Rules:
- Answer using ONLY the information in the provided context.
- If the context does not contain the answer, say so plainly instead of guessing.
- Be concise and technical; use equations or variable names exactly as they appear in the context.
- When helpful, cite the source paper using the `source_file` shown in the context."""


def chat(question, retriever, llm, top_k=5):
    """Answer a natural-language question about triple-axis spectrometers using retrieved context."""
    # Retrieve relevant passages from the knowledge base
    results = retriever.retrieve(question, top_k=top_k)
    if not results:
        return "No relevant information found in the knowledge base."

    # Build context with source attribution so the model can cite papers
    context = "\n\n".join(
        f"[Source: {doc['metadata'].get('source_file', 'unknown')}]\n{doc['content']}"
        for doc in results
    )

    response = llm.invoke([
        SystemMessage(QA_SYSTEM_PROMPT),
        HumanMessage(f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),
    ])
    return response.content.strip()

In [13]:
answer = chat("what is resolution matrix?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'what is resolution matrix?'
Top K: 5, score threshold: 0.0
generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.54it/s]

Generated embeddings with shape : (1, 384)
Retrieved 5 documents (after filtering)


The resolution matrix, $M$, is a matrix used in computing the resolution function, $R(X)$.

Key details regarding $M$:

*   **Function:** All information about the resolution function is contained in the three eigenvalues of $M$, the angle $\theta$, and the efficiency factor $R_0$.
*   **Inverse:** The inverse of $M$, denoted $M^{-1}$, is the covariance matrix $\{(X, X_j)\}$ of the resolution function.
*   **Form:** $M$ can be expressed in terms of the three components of $K$, where one component ($z$) is not coupled to the other two.
*   **Usage:** The resolution function $R(X)$ is given by the form:
    $$R(X) = R_0(2z_c - z_l/d) M \exp \left(-\frac{1}{2}X^T M X\right) \quad (1)$$
    where $X$ is the column matrix of the four-component vector $X = (Q - Q_o, \omega - \omega_o)$.

(Source: Popvici 1975 part4.pdf; Mitchell - 1984.pdf)


In [20]:
### Generate triple-axis scan commands from a natural-language request

SCAN_SYSTEM_PROMPT = """You are an assistant that converts natural-language requests \
into Triple Axis instrument scan commands.

Rules:
- Use ONLY the command syntax shown in the provided context.
- Output ONLY the command(s), one per line. No explanation, no markdown, no code fences.
- If a monitor count (preset) or scan title is requested, emit those commands first.
- For a reciprocal-space scan use: scan h <start> <stop> <step> k <k> l <l> e <e>
- If the request is missing a value, use 0 for that coordinate.
- If the request contain typos, reply the mistake and suggest a fix based on the documented commands."""


def generate_scan_command(request, retriever, llm, top_k=3):
    """Translate a natural-language request into triple-axis scan command(s)."""
    # Retrieve the command syntax from the knowledge base
    results = retriever.retrieve(request, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No command syntax found in the knowledge base."

    response = llm.invoke([
        SystemMessage(SCAN_SYSTEM_PROMPT),
        HumanMessage(f"Command reference:\n{context}\n\nRequest: {request}\n\nCommands:"),
    ])
    return response.content.strip()


In [23]:
bragg_peaks = ([0, 0, 1], [1, 2, 1], [3, 2, 1])
# Example
cmd = generate_scan_command(
    f"move to the following Bragg peak positions {bragg_peaks}, and generate a command with appropriate titles of scanning h plus minus 0.5 relative to the Bragg peak position, step size 0.1, preset mcu 100 for each Bragg peak",
    rag_retriever,
    llm,
)
print(cmd)

Retrieving documents for query: 'move to the following Bragg peak positions ([0, 0, 1], [1, 2, 1], [3, 2, 1]), and generate a command with appropriate titles of scanning h plus minus 0.5 relative to the Bragg peak position, step size 0.1, preset mcu 100 for each Bragg peak'
Top K: 3, score threshold: 0.0
generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.68it/s]

Generated embeddings with shape : (1, 384)
Retrieved 2 documents (after filtering)


preset mcu 100
scantitle "along h"
br [0, 0, 1]
scan h -0.5 0.5 0.1 k 0 l 1 e 0
br [1, 2, 1]
scan h 0.5 1.5 0.1 k 2 l 1 e 0
br [3, 2, 1]
scan h 2.5 3.5 0.1 k 2 l 1 e 0


In [18]:
bragg_peaks = ([0, 0, 1], [1, 2, 1], [3, 2, 1])
# Example
cmd = generate_scan_command(
    f"move to the following Bragg peak positions {bragg_peaks}, and generate a command with appropriate titles of inelastic scans from 0 to 10 meV searching for spin gaps, preset mcu 100 for each Bragg peak",
    rag_retriever,
    llm,
)
print(cmd)

Retrieving documents for query: 'move to the following Bragg peak positions ([0, 0, 1], [1, 2, 1], [3, 2, 1]), and generate a command with appropriate titles of inelastic scans from 0 to 10 meV searching for spin gaps, preset mcu 100 for each Bragg peak'
Top K: 3, score threshold: 0.0
generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.34it/s]

Generated embeddings with shape : (1, 384)
Retrieved 2 documents (after filtering)


preset mcu 100
scantitle "Spin Gap Search"
br 0 0 1
scan h 0 k 0 l 0 e 0 10 0.1
br 1 2 1
scan h 0 k 0 l 0 e 0 10 0.1
br 3 2 1
scan h 0 k 0 l 0 e 0 10 0.1


In [21]:
# Example
cmd = generate_scan_command(
    f"Are there any typos in these scan commands: scan h 0 1 0.1 h 0 k 0 e 1, scan h 1 k 1 l 0 e 1 2",
    rag_retriever,
    llm,
)
print(cmd)

Retrieving documents for query: 'Are there any typos in these scan commands: scan h 0 1 0.1 h 0 k 0 e 1, scan h 1 k 1 l 0 e 1 2'
Top K: 3, score threshold: 0.0
generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.35it/s]

Generated embeddings with shape : (1, 384)
Retrieved 2 documents (after filtering)


The command `scan h 0 1 0.1 h 0 k 0 e 1` contains structural errors. You cannot repeat the coordinate definition (e.g., `h`) within the scan command. If you intended to scan h, k, and e, the format should be `scan h <start> <stop> <step> k <k> l <l> e <start> <stop> <step>`.

The command `scan h 1 k 1 l 0 e 1 2` is missing required start, stop, and step values for most coordinates, making the syntax invalid.

Please provide the intended scan parameters (start, stop, step) for all coordinates you wish to include.
